# Diffusion Model Implementation

This notebook implements a diffusion model for image generation using forward and reverse diffusion processes.

## Overview

Diffusion models work by:
1. **Forward Diffusion**: Gradually adding noise to images over T timesteps
2. **Reverse Diffusion**: Learning to denoise and generate images by reversing the forward process

The model learns to predict the noise added at each step, allowing it to generate new images from random noise.

## 1. Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Forward Diffusion Process

The forward diffusion process gradually adds Gaussian noise to an image over T timesteps.

At each timestep t, we sample:
$$x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon$$

where:
- $x_0$ is the original image
- $\epsilon \sim \mathcal{N}(0, I)$ is random noise
- $\bar{\alpha}_t$ controls the noise schedule

In [ ]:
def linear_beta_schedule(timesteps, beta_start=0.0001, beta_end=0.02):
    """
    Linear schedule for beta values.
    
    Args:
        timesteps: Number of diffusion steps
        beta_start: Starting beta value
        beta_end: Ending beta value
    
    Returns:
        beta values for each timestep
    """
    return torch.linspace(beta_start, beta_end, timesteps)


def cosine_beta_schedule(timesteps, s=0.008):
    """
    Cosine schedule for beta values (better for small images).
    
    Args:
        timesteps: Number of diffusion steps
        s: Small offset to prevent beta from being too small
    
    Returns:
        beta values for each timestep
    """
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * torch.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.9999)


class ForwardDiffusion:
    """
    Implements the forward diffusion process.
    """
    
    def __init__(self, timesteps=1000, beta_schedule='linear'):
        """
        Args:
            timesteps: Number of diffusion steps
            beta_schedule: 'linear' or 'cosine'
        """
        self.timesteps = timesteps
        
        # Define beta schedule
        if beta_schedule == 'linear':
            self.betas = linear_beta_schedule(timesteps)
        elif beta_schedule == 'cosine':
            self.betas = cosine_beta_schedule(timesteps)
        else:
            raise ValueError(f"Unknown beta schedule: {beta_schedule}")
        
        # Pre-calculate useful values
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        
        # Calculations for diffusion q(x_t | x_0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        
        # Calculations for posterior q(x_{t-1} | x_t, x_0)
        self.posterior_variance = (
            self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
    
    def q_sample(self, x_start, t, noise=None):
        """
        Forward diffusion: Sample x_t from x_0 and t.
        
        Args:
            x_start: Original images (x_0)
            t: Timestep (batch of integers)
            noise: Optional pre-generated noise
        
        Returns:
            Noisy images at timestep t
        """
        if noise is None:
            noise = torch.randn_like(x_start)
        
        sqrt_alphas_cumprod_t = self.sqrt_alphas_cumprod[t].reshape(-1, 1, 1, 1)
        sqrt_one_minus_alphas_cumprod_t = self.sqrt_one_minus_alphas_cumprod[t].reshape(-1, 1, 1, 1)
        
        # Apply noise: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
        return sqrt_alphas_cumprod_t * x_start + sqrt_one_minus_alphas_cumprod_t * noise, noise
    
    def get_loss_weight(self, t):
        """
        Get loss weighting for different timesteps (optional).
        """
        return torch.ones_like(t, dtype=torch.float32)

## 3. Simple U-Net Architecture

A simplified U-Net architecture for the denoising model. This network learns to predict the noise added at each timestep.

In [ ]:
class SimpleUNet(nn.Module):
    """
    Simplified U-Net for denoising.
    """
    
    def __init__(self, in_channels=1, out_channels=1, time_emb_dim=128):
        super().__init__()
        
        # Time embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.ReLU()
        )
        
        # Encoder (downsampling)
        self.conv1 = DoubleConv(in_channels, 64, time_emb_dim)
        self.pool1 = nn.MaxPool2d(2)
        
        self.conv2 = DoubleConv(64, 128, time_emb_dim)
        self.pool2 = nn.MaxPool2d(2)
        
        self.conv3 = DoubleConv(128, 256, time_emb_dim)
        self.pool3 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = DoubleConv(256, 512, time_emb_dim)
        
        # Decoder (upsampling)
        self.upconv3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.conv4 = DoubleConv(512, 256, time_emb_dim)
        
        self.upconv2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv5 = DoubleConv(256, 128, time_emb_dim)
        
        self.upconv1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv6 = DoubleConv(128, 64, time_emb_dim)
        
        # Output
        self.out = nn.Conv2d(64, out_channels, 1)
    
    def forward(self, x, t):
        # Time embedding
        t_emb = self.time_mlp(t)
        
        # Encoder
        x1 = self.conv1(x, t_emb)
        x = self.pool1(x1)
        
        x2 = self.conv2(x, t_emb)
        x = self.pool2(x2)
        
        x3 = self.conv3(x, t_emb)
        x = self.pool3(x3)
        
        # Bottleneck
        x = self.bottleneck(x, t_emb)
        
        # Decoder with skip connections
        x = self.upconv3(x)
        x = torch.cat([x, x3], dim=1)
        x = self.conv4(x, t_emb)
        
        x = self.upconv2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv5(x, t_emb)
        
        x = self.upconv1(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv6(x, t_emb)
        
        return self.out(x)


class DoubleConv(nn.Module):
    """Double convolution block with time embedding."""
    
    def __init__(self, in_channels, out_channels, time_emb_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, out_channels),
            nn.ReLU()
        )
    
    def forward(self, x, t_emb):
        x = self.conv(x)
        time_emb = self.time_mlp(t_emb)[:, :, None, None]
        return x + time_emb


class SinusoidalPositionEmbeddings(nn.Module):
    """Sinusoidal position embeddings for timesteps."""
    
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    
    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = np.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

## 4. Reverse Diffusion Process

The reverse diffusion process uses the trained model to denoise images step by step, starting from random noise.

In [ ]:
class ReverseDiffusion:
    """
    Implements the reverse diffusion process (sampling/generation).
    """
    
    def __init__(self, model, forward_diffusion, device='cpu'):
        """
        Args:
            model: Trained denoising model
            forward_diffusion: ForwardDiffusion object for noise schedule
            device: Device to run on
        """
        self.model = model
        self.forward_diffusion = forward_diffusion
        self.device = device
        
        # Move noise schedule parameters to device
        self.betas = forward_diffusion.betas.to(device)
        self.alphas = forward_diffusion.alphas.to(device)
        self.alphas_cumprod = forward_diffusion.alphas_cumprod.to(device)
        self.sqrt_alphas_cumprod = forward_diffusion.sqrt_alphas_cumprod.to(device)
        self.sqrt_one_minus_alphas_cumprod = forward_diffusion.sqrt_one_minus_alphas_cumprod.to(device)
        self.posterior_variance = forward_diffusion.posterior_variance.to(device)
    
    @torch.no_grad()
    def p_sample(self, x, t, t_index):
        """
        Sample x_{t-1} from x_t using the model.
        
        Args:
            x: Current noisy image (x_t)
            t: Current timestep
            t_index: Index of timestep (for indexing schedules)
        
        Returns:
            Denoised image at t-1
        """
        # Predict noise
        predicted_noise = self.model(x, t)
        
        # Get schedule values
        betas_t = self.betas[t_index]
        sqrt_one_minus_alphas_cumprod_t = self.sqrt_one_minus_alphas_cumprod[t_index]
        sqrt_recip_alphas_t = torch.sqrt(1.0 / self.alphas[t_index])
        
        # Compute mean of p(x_{t-1} | x_t)
        model_mean = sqrt_recip_alphas_t * (
            x - betas_t * predicted_noise / sqrt_one_minus_alphas_cumprod_t
        )
        
        if t_index == 0:
            return model_mean
        else:
            posterior_variance_t = self.posterior_variance[t_index]
            noise = torch.randn_like(x)
            return model_mean + torch.sqrt(posterior_variance_t) * noise
    
    @torch.no_grad()
    def sample(self, shape, return_all_steps=False):
        """
        Generate images by sampling from noise.
        
        Args:
            shape: Shape of images to generate (batch_size, channels, height, width)
            return_all_steps: If True, return all intermediate steps
        
        Returns:
            Generated images (and all steps if return_all_steps=True)
        """
        batch_size = shape[0]
        self.model.eval()
        
        # Start from pure noise
        img = torch.randn(shape, device=self.device)
        imgs = []
        
        # Iteratively denoise
        timesteps = self.forward_diffusion.timesteps
        for i in tqdm(reversed(range(timesteps)), desc='Sampling', total=timesteps):
            t = torch.full((batch_size,), i, device=self.device, dtype=torch.long)
            img = self.p_sample(img, t, i)
            
            if return_all_steps:
                imgs.append(img.cpu())
        
        if return_all_steps:
            return img, imgs
        return img
    
    @torch.no_grad()
    def sample_fast(self, shape, steps=50):
        """
        Fast sampling using fewer steps (DDIM-style).
        
        Args:
            shape: Shape of images to generate
            steps: Number of sampling steps (fewer than training timesteps)
        
        Returns:
            Generated images
        """
        batch_size = shape[0]
        self.model.eval()
        
        # Start from pure noise
        img = torch.randn(shape, device=self.device)
        
        # Use evenly spaced timesteps
        timesteps = self.forward_diffusion.timesteps
        step_size = timesteps // steps
        time_steps = list(range(0, timesteps, step_size))[::-1]
        
        for i in tqdm(time_steps, desc='Fast Sampling'):
            t = torch.full((batch_size,), i, device=self.device, dtype=torch.long)
            img = self.p_sample(img, t, i)
        
        return img

## 5. Training Function

In [ ]:
def train_epoch(model, dataloader, optimizer, forward_diffusion, device):
    """
    Train the model for one epoch.
    
    Args:
        model: Denoising model
        dataloader: Training data loader
        optimizer: Optimizer
        forward_diffusion: ForwardDiffusion object
        device: Device to train on
    
    Returns:
        Average loss for the epoch
    """
    model.train()
    total_loss = 0
    
    for batch_idx, (images, _) in enumerate(tqdm(dataloader, desc='Training')):
        images = images.to(device)
        batch_size = images.shape[0]
        
        # Sample random timesteps
        t = torch.randint(0, forward_diffusion.timesteps, (batch_size,), device=device).long()
        
        # Add noise to images
        noise = torch.randn_like(images)
        noisy_images, _ = forward_diffusion.q_sample(images, t, noise)
        
        # Predict noise
        predicted_noise = model(noisy_images, t)
        
        # Compute loss (MSE between predicted and actual noise)
        loss = F.mse_loss(predicted_noise, noise)
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

## 6. Demonstration and Visualization

Let's demonstrate the diffusion model on MNIST dataset.

In [ ]:
# Configuration
TIMESTEPS = 300
IMAGE_SIZE = 28
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 1e-3

# Load MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Normalize to [-1, 1]
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print(f"Dataset size: {len(train_dataset)}")
print(f"Number of batches: {len(train_loader)}")

In [ ]:
# Initialize model and diffusion processes
forward_diffusion = ForwardDiffusion(timesteps=TIMESTEPS, beta_schedule='cosine')
model = SimpleUNet(in_channels=1, out_channels=1, time_emb_dim=128).to(device)

# Count parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {num_params:,}")

# Initialize optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

### Visualize Forward Diffusion Process

In [ ]:
# Visualize forward diffusion on a sample image
sample_image, _ = train_dataset[0]
sample_image = sample_image.unsqueeze(0).to(device)

# Show noise addition at different timesteps
timesteps_to_show = [0, 50, 100, 150, 200, 250, 299]
fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(15, 2))

for idx, t in enumerate(timesteps_to_show):
    t_tensor = torch.tensor([t], device=device)
    noisy_image, _ = forward_diffusion.q_sample(sample_image, t_tensor)
    
    axes[idx].imshow(noisy_image.squeeze().cpu().numpy(), cmap='gray')
    axes[idx].set_title(f't={t}')
    axes[idx].axis('off')

plt.suptitle('Forward Diffusion Process')
plt.tight_layout()
plt.show()

### Train the Model

In [ ]:
# Training loop
losses = []

for epoch in range(EPOCHS):
    avg_loss = train_epoch(model, train_loader, optimizer, forward_diffusion, device)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {avg_loss:.4f}")
    
    # Generate samples every few epochs
    if (epoch + 1) % 5 == 0:
        reverse_diffusion = ReverseDiffusion(model, forward_diffusion, device)
        samples = reverse_diffusion.sample((16, 1, IMAGE_SIZE, IMAGE_SIZE))
        
        # Visualize generated samples
        fig, axes = plt.subplots(4, 4, figsize=(8, 8))
        for i in range(16):
            ax = axes[i // 4, i % 4]
            ax.imshow(samples[i].squeeze().cpu().numpy(), cmap='gray')
            ax.axis('off')
        plt.suptitle(f'Generated Samples (Epoch {epoch+1})')
        plt.tight_layout()
        plt.show()

# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)
plt.show()

### Generate New Images

In [ ]:
# Generate final samples
reverse_diffusion = ReverseDiffusion(model, forward_diffusion, device)
generated_images = reverse_diffusion.sample((25, 1, IMAGE_SIZE, IMAGE_SIZE))

# Visualize
fig, axes = plt.subplots(5, 5, figsize=(12, 12))
for i in range(25):
    ax = axes[i // 5, i % 5]
    ax.imshow(generated_images[i].squeeze().cpu().numpy(), cmap='gray')
    ax.axis('off')

plt.suptitle('Final Generated Images', fontsize=16)
plt.tight_layout()
plt.show()

### Visualize Reverse Diffusion Process

In [ ]:
# Generate image and save all intermediate steps
final_image, all_steps = reverse_diffusion.sample(
    (1, 1, IMAGE_SIZE, IMAGE_SIZE), 
    return_all_steps=True
)

# Show reverse diffusion at different timesteps
steps_to_show = [0, 50, 100, 150, 200, 250, 299]
fig, axes = plt.subplots(1, len(steps_to_show), figsize=(15, 2))

for idx, step_idx in enumerate(steps_to_show):
    img = all_steps[step_idx]
    axes[idx].imshow(img.squeeze().numpy(), cmap='gray')
    axes[idx].set_title(f't={TIMESTEPS - step_idx - 1}')
    axes[idx].axis('off')

plt.suptitle('Reverse Diffusion Process (Denoising)')
plt.tight_layout()
plt.show()

### Fast Sampling Demo

In [ ]:
# Compare standard vs fast sampling
print("Generating with standard sampling...")
standard_samples = reverse_diffusion.sample((4, 1, IMAGE_SIZE, IMAGE_SIZE))

print("Generating with fast sampling (50 steps)...")
fast_samples = reverse_diffusion.sample_fast((4, 1, IMAGE_SIZE, IMAGE_SIZE), steps=50)

# Visualize comparison
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for i in range(4):
    axes[0, i].imshow(standard_samples[i].squeeze().cpu().numpy(), cmap='gray')
    axes[0, i].set_title(f'Standard {i+1}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(fast_samples[i].squeeze().cpu().numpy(), cmap='gray')
    axes[1, i].set_title(f'Fast {i+1}')
    axes[1, i].axis('off')

plt.suptitle('Standard (300 steps) vs Fast Sampling (50 steps)')
plt.tight_layout()
plt.show()

## 7. Save and Load Model

In [ ]:
# Save the trained model
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'timesteps': TIMESTEPS,
    'losses': losses
}, 'diffusion_model.pth')

print("Model saved to 'diffusion_model.pth'")

In [ ]:
# Load the model (example)
# checkpoint = torch.load('diffusion_model.pth')
# model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
# print("Model loaded successfully")

## Summary

This notebook implements a complete diffusion model with:

1. **Forward Diffusion**: Gradually adds noise to images using a noise schedule (linear or cosine)
2. **Reverse Diffusion**: Learns to denoise images step by step to generate new samples
3. **U-Net Architecture**: A neural network that predicts the noise at each timestep
4. **Training**: Trains the model to predict noise using MSE loss
5. **Sampling**: Generates new images from random noise
6. **Fast Sampling**: Accelerated generation using fewer steps

### Key Concepts:

- The model learns to reverse the diffusion process by predicting noise
- Time embeddings help the network understand which timestep it's processing
- The noise schedule controls how quickly noise is added/removed
- U-Net architecture with skip connections helps preserve spatial information

### Extensions:

- Try different noise schedules (cosine often works better for small images)
- Experiment with different architectures (attention mechanisms, transformers)
- Implement conditional generation (class-conditional, text-to-image)
- Use more sophisticated sampling methods (DDIM, DPM-Solver)
- Scale to higher resolution images with progressive training